# LoRA fine-tune Qwen2.5-Coder-1.5B on ERCOT text-to-SQL

Runs on Kaggle P100 or T4 (free tier). Expected wall-clock: ~15-30 min for 280 training pairs at 3 epochs.

Uses **plain LoRA in fp16** — not QLoRA in 4-bit — for portability across older GPUs (P100 lacks bitsandbytes support). 1.5B model in fp16 fits comfortably in 16 GB VRAM.

**Before running:** attach your dataset (uploaded JSONL files) as Kaggle Dataset input.

## 1. Install

In [ ]:
!pip install -q -U transformers peft trl accelerate datasets torchao

## 2. Auth — HuggingFace token

Add `HF_TOKEN` as a Kaggle Secret (Add-ons → Secrets) and enable it for this notebook.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

login(token=UserSecretsClient().get_secret('HF_TOKEN'))

## 3. Config (mirror of training/config.py)

In [ ]:
BASE_MODEL = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
ADAPTER_REPO = 'visethchapman/ercot-text2sql-qwen-1.5b-lora'

LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LORA_TARGETS = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']

LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
BATCH_SIZE = 2           # P100/T4 friendly
GRAD_ACCUM = 8           # effective batch = 16
MAX_SEQ_LEN = 1024       # training pairs are short; larger wastes memory

# Kaggle mount path — check the Input sidebar in the notebook UI.
# Standard form is /kaggle/input/<dataset-slug>, but under some accounts
# it appears as /kaggle/input/datasets/<user>/<slug>.
DATASET_PATH = '/kaggle/input/datasets/visethsean/ercot-text2sql-v1'

## 4. Load model in 4-bit (QLoRA)

In [ ]:
import os
# Hide GPU 1 before torch loads. Kaggle's T4 x2 tempts Accelerate into splitting
# the batch across both cards, then loss compute crashes on cuda:0 vs cuda:1
# tensor mismatch. Restart the kernel if you're editing this cell — the env
# var only takes effect before the first torch import.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Plain fp16 load (no 4-bit quantization). 1.5B in fp16 = ~3GB VRAM.
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map={'': 0},
)

## 5. Attach LoRA adapter

In [ ]:
from peft import LoraConfig, get_peft_model

lora = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS,
    bias='none', task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 6. Load dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset('json', data_files={
    'train': f'{DATASET_PATH}/train.jsonl',
    'val':   f'{DATASET_PATH}/val.jsonl',
})
print(ds)
print('Sample train row:', ds['train'][0])

## 7. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

cfg = SFTConfig(
    output_dir='/kaggle/working/adapter',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    max_length=MAX_SEQ_LEN,           # TRL 0.12+ renamed from max_seq_length
    logging_steps=10,
    save_steps=100,
    eval_strategy='steps',
    eval_steps=100,
    fp16=True,                        # P100/T4 do not support bf16 in hardware
    report_to='none',
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds['train'],
    eval_dataset=ds['val'],
    args=cfg,
    processing_class=tokenizer,        # TRL 0.12+ renamed from tokenizer
)
trainer.train()

## 8. Save + push adapter to HF Hub

In [ ]:
trainer.save_model('/kaggle/working/adapter')
model.push_to_hub(ADAPTER_REPO, private=False)
tokenizer.push_to_hub(ADAPTER_REPO)
print(f'Adapter pushed: https://huggingface.co/{ADAPTER_REPO}')

## 9. Smoke test

In [ ]:
# Use the SAME system prompt the model saw during training (see dataset/format.py).
# A stub prompt lets the model hallucinate tables — it wasn't trained to memorize
# schema, only to map (schema in prompt + question) → SQL.
SCHEMA = """eia.demand(region, period, value, value_units) — region='ERCO'; period in UTC; value in MWh
noaa.daily_weather(station_id, obs_date, tmax_c, tmin_c, prcp_mm, awnd_ms) — Houston station; obs_date is local date
noaa.stations(station_id, name, state, nearest_eia_region)

Notes: All demand is UTC. Houston weather is local date. For joins,
cast period to local date: (period AT TIME ZONE 'America/Chicago')::date"""

SYSTEM = f"""You are a Postgres SQL expert for ERCOT electricity-demand and Houston weather data.

Schema:
{SCHEMA}

Return ONLY valid Postgres SQL. No explanation, no markdown fences."""

test_q = 'What was peak hourly ERCOT demand in 2024?'
msgs = [
    {'role': 'system', 'content': SYSTEM},
    {'role': 'user', 'content': test_q},
]
inputs = tokenizer.apply_chat_template(
    msgs, return_tensors='pt', add_generation_prompt=True, return_dict=True,
).to(model.device)
out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
in_len = inputs['input_ids'].shape[1]
print(tokenizer.decode(out[0][in_len:], skip_special_tokens=True))